# 6b — Regressions: Clean Method

**Optional notebook** — repeats the `CLEAN/6_Regressions_Unified.ipynb` specification 
against the FINAL CODE RECAP pipeline data.

| Model | Spec | DV |
|---|---|---|
| 1 | Kitchen-sink OLS | log ECI |
| 2 | AR baseline | ECI |
| 3a | Parsimonious, no lag | ECI |
| 3b | Parsimonious + ECI(t-1) | ECI |
| 3c | All vars lagged | ECI |
| 3d | Extended controls + ECI(t-1) | ECI |
| 3e | First differences (error-correction) | ΔECI |

All models use clustered SEs by country.  
Output: `Graphics/NB6b/`

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

OUT = os.path.join('Graphics', 'NB6b')
os.makedirs(OUT, exist_ok=True)

ECI_COL = 'Economic Complexity Index'
FONT    = 'IBM Plex Sans, Arial, sans-serif'
BG      = '#fafafa'
NAVY    = '#1a1a2e'
GRID    = '#e0e0e0'

def save_html(fig, name, cdn=False):
    path = os.path.join(OUT, f'{name}.html')
    fig.write_html(
        path,
        include_plotlyjs='cdn' if cdn else True,
        config={'displayModeBar': False, 'responsive': True},
    )
    print(f'Saved: {path}')

print('Setup complete')

Setup complete


## 1  Load data

In [2]:
master   = pd.read_csv('intermediary/Master.csv', dtype={'Country Code': str})
clusters = pd.read_csv('intermediary/clustersagg.csv', dtype={'Country Code': str})

cluster_map = clusters[['Country Code', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')
master = master.merge(cluster_map, on='Country Code', how='inner')

master['Year'] = master['Year'].astype(int)
master = master.sort_values(['Country Code', 'Year']).reset_index(drop=True)

# ── 54-country include_list (same as NB4/NB5) ─────────────────────────────────
INCLUDE = [
    'AGO','ARE','AZE','BFA','BHR','BOL','CHL','CIV','CMR',
    'COD','COG','DZA','ECU','EGY','ETH','GAB','GHA','GIN',
    'GNQ','IDN','IRN','IRQ','KAZ','KEN','KWT','LAO','LBR',
    'LBY','MDG','MLI','MMR','MNG','MOZ','MWI','MYS','NER',
    'NGA','OMN','PNG','QAT','RUS','RWA','SAU','TCD','TGO',
    'TTO','TZA','UGA','UZB','VEN','VNM','YEM','ZMB','ZWE',
]
master = master[master['Country Code'].isin(INCLUDE)].copy()

print(f'Sample: {master["Country Code"].nunique()} countries, {len(master):,} obs')
print(f'Years: {master["Year"].min()}–{master["Year"].max()}')
print(f'\nCluster distribution:')
print(master.drop_duplicates('Country Code')['ClusterLabels'].value_counts())

Sample: 54 countries, 1,350 obs
Years: 1995–2019

Cluster distribution:
ClusterLabels
No Oil, No Minerals      23
Some Oil, No Minerals    15
Oil, Few Minerals         9
Minerals, No Oil          7
Name: count, dtype: int64


## 2  Feature engineering

In [3]:
df = master.copy()

df['Total_Production_Value_Per_Capita'] = (
    df['Total_Production_Value'] / df['Population'].replace(0, np.nan)
)

df['log_HCI']              = np.log1p(df['Human capital index'].clip(lower=0))
df['log_GFCF']             = np.log1p(
    df['Gross fixed capital formation, all, Constant prices, Percent of GDP'].clip(lower=0))
df['log_Production_Value'] = np.log1p(df['Total_Production_Value_Per_Capita'].clip(lower=0))

eci_min      = df[ECI_COL].min()
df['log_ECI'] = np.log(df[ECI_COL] - eci_min + 1)

df['ECI_lag1']  = df.groupby('Country Code')[ECI_COL].shift(1)
df['delta_ECI'] = df[ECI_COL] - df['ECI_lag1']

BASE_INDEP = [
    'log_HCI', 'log_GFCF',
    'Political stability — estimate',
    'Rule of law index',
    'log_Production_Value',
    'Trade (% of GDP)',
]
EXTRA_CONTROLS = [
    'Hydrocarbons_Dominant',
    'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant',
    'Access to electricity (% of population)',
]

for var in BASE_INDEP:
    df[f'{var}_lag1'] = df.groupby('Country Code')[var].shift(1)

# Mean-centred interactions (levels)
hci_c  = df['log_HCI']             - df['log_HCI'].mean()
gfcf_c = df['log_GFCF']            - df['log_GFCF'].mean()
prod_c = df['log_Production_Value'] - df['log_Production_Value'].mean()
df['log_HCI_x_log_Production']  = hci_c  * prod_c
df['log_GFCF_x_log_Production'] = gfcf_c * prod_c

# Mean-centred interactions (lagged)
hci_l_c  = df['log_HCI_lag1']             - df['log_HCI_lag1'].mean()
gfcf_l_c = df['log_GFCF_lag1']            - df['log_GFCF_lag1'].mean()
prod_l_c = df['log_Production_Value_lag1'] - df['log_Production_Value_lag1'].mean()
df['log_HCI_x_log_Production_lag1']  = hci_l_c  * prod_l_c
df['log_GFCF_x_log_Production_lag1'] = gfcf_l_c * prod_l_c

# First differences (for Model 3e)
for var in BASE_INDEP:
    df[f'd_{var}'] = df.groupby('Country Code')[var].diff()
df['d_Electricity'] = df.groupby('Country Code')['Access to electricity (% of population)'].diff()
df['d_log_HCI_x_log_Production']  = df.groupby('Country Code')['log_HCI_x_log_Production'].diff()
df['d_log_GFCF_x_log_Production'] = df.groupby('Country Code')['log_GFCF_x_log_Production'].diff()

print('Feature engineering complete')
print(f'Shape: {df.shape}')

Feature engineering complete
Shape: (1350, 77)


## 3  OLS helpers

In [4]:
def run_ols(df_in, dv, regressors, cluster_by='Country Code'):
    req = [dv] + regressors + [cluster_by]
    sub = df_in.dropna(subset=req).copy()
    X   = sm.add_constant(sub[regressors])
    y   = sub[dv]
    res = sm.OLS(y, X).fit(
        cov_type='cluster',
        cov_kwds={'groups': sub[cluster_by]},
    )
    return res, sub


def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''


def to_tbl(res):
    tbl = pd.DataFrame({
        'Variable': res.params.index,
        'Coef':     res.params.values,
        'SE':       res.bse.values,
        'p':        res.pvalues.values,
        'CI_lo':    res.conf_int().iloc[:, 0].values,
        'CI_hi':    res.conf_int().iloc[:, 1].values,
    })
    tbl['Stars'] = tbl['p'].apply(stars)
    return tbl


ALL_RESULTS = {}
print('Helpers defined')

Helpers defined


## 4  Run models

In [5]:
# ── Model 1 — Kitchen sink (log ECI, cluster SE by Cluster) ──────────────────
KITCHEN_SINK_VARS = [
    'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)',
    'Agriculture', 'Capital depreciation rate', 'Clientelism index',
    'Death rates, crude per 1000 people',
    'Domestic credit to private sector (% of GDP)',
    'GDP per capita (constant prices, PPP)', 'Government revenue',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Human capital index', 'Industry',
    'Inflation, consumer prices (annual %)', 'Landlocked',
    'Lending interest rate (%)', 'Life expectancy at birth, total (years)',
    'Manufacturing', 'Mineral rents (% of GDP)',
    'Mobile cellular subscriptions (per 100 people)',
    'Natural gas rents (% of GDP)', 'Oil rents (% of GDP)',
    'Political corruption index', 'Political stability — estimate',
    'Primary net lending, General government, Percent of GDP',
    'Property rights', 'Real interest rate (%)', 'Rule of law index',
    'Services', 'Share of consumption in GDP',
    'Share of government spending in GDP', 'Share of investment in GDP',
    'Total natural resources rents (% of GDP)', 'Trade (% of GDP)',
    'Urban population (% of total population)',
    'Use of IMF credit (DOD, current US$)',
    'Total_Production_Value_Per_Capita', 'Total_Reserves_Value_Per_Capita',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant', 'Precious_Metals_Dominant',
]
KITCHEN_SINK_VARS = [v for v in KITCHEN_SINK_VARS if v in df.columns]

res_m1, _ = run_ols(df, 'log_ECI', KITCHEN_SINK_VARS, 'Cluster')
ALL_RESULTS['M1'] = (res_m1, to_tbl(res_m1),
                     'log ECI', 'Kitchen-sink OLS')
print(f'M1  N={int(res_m1.nobs):,}  R²={res_m1.rsquared:.4f}')

M1  N=1,290  R²=0.5953


In [6]:
# ── Model 2 — AR baseline ─────────────────────────────────────────────────────
res_m2, _ = run_ols(df, ECI_COL, ['ECI_lag1'])
ALL_RESULTS['M2'] = (res_m2, to_tbl(res_m2), 'ECI', 'AR baseline')
print(f'M2  N={int(res_m2.nobs):,}  R²={res_m2.rsquared:.4f}')

M2  N=1,296  R²=0.7709


In [7]:
# ── Model 3a — Parsimonious, no lag ──────────────────────────────────────────
VARS_3A = BASE_INDEP + ['log_HCI_x_log_Production', 'log_GFCF_x_log_Production']
res_3a, _ = run_ols(df, ECI_COL, VARS_3A)
ALL_RESULTS['3a'] = (res_3a, to_tbl(res_3a), 'ECI', 'Parsimonious, no lag')
print(f'3a  N={int(res_3a.nobs):,}  R²={res_3a.rsquared:.4f}')

# ── Model 3b — Parsimonious + ECI(t-1) ───────────────────────────────────────
VARS_3B = BASE_INDEP + ['ECI_lag1', 'log_HCI_x_log_Production', 'log_GFCF_x_log_Production']
res_3b, _ = run_ols(df, ECI_COL, VARS_3B)
ALL_RESULTS['3b'] = (res_3b, to_tbl(res_3b), 'ECI', 'Parsimonious + ECI(t-1)')
print(f'3b  N={int(res_3b.nobs):,}  R²={res_3b.rsquared:.4f}')

# ── Model 3c — All vars lagged ────────────────────────────────────────────────
LAGGED_BASE = [f'{v}_lag1' for v in BASE_INDEP]
VARS_3C = LAGGED_BASE + [
    'ECI_lag1',
    'log_HCI_x_log_Production_lag1',
    'log_GFCF_x_log_Production_lag1',
]
res_3c, _ = run_ols(df, ECI_COL, VARS_3C)
ALL_RESULTS['3c'] = (res_3c, to_tbl(res_3c), 'ECI', 'All vars lagged')
print(f'3c  N={int(res_3c.nobs):,}  R²={res_3c.rsquared:.4f}')

# ── Model 3d — Extended controls + ECI(t-1) ──────────────────────────────────
VARS_3D = BASE_INDEP + EXTRA_CONTROLS + [
    'ECI_lag1',
    'log_HCI_x_log_Production',
    'log_GFCF_x_log_Production',
]
res_3d, _ = run_ols(df, ECI_COL, VARS_3D)
ALL_RESULTS['3d'] = (res_3d, to_tbl(res_3d), 'ECI', 'Extended controls')
print(f'3d  N={int(res_3d.nobs):,}  R²={res_3d.rsquared:.4f}')

# ── Model 3e — First differences (error-correction) ──────────────────────────
VARS_3E = [f'd_{v}' for v in BASE_INDEP] + [
    'd_Electricity', 'ECI_lag1',
    'd_log_HCI_x_log_Production',
    'd_log_GFCF_x_log_Production',
]
res_3e, _ = run_ols(df, 'delta_ECI', VARS_3E)
ALL_RESULTS['3e'] = (res_3e, to_tbl(res_3e), 'ΔECI', 'First differences')
print(f'3e  N={int(res_3e.nobs):,}  R²={res_3e.rsquared:.4f}')

3a  N=1,324  R²=0.3590
3b  N=1,273  R²=0.7848
3c  N=1,270  R²=0.7884
3d  N=1,264  R²=0.7932


3e  N=1,268  R²=0.0740


## 5  Formatted regression table

In [8]:
DISPLAY_LABELS = {
    'const':                                        'Constant',
    'log_HCI':                                      'Human Capital (log)',
    'log_GFCF':                                     'GFCF (log)',
    'Political stability — estimate':               'Political Stability',
    'Rule of law index':                            'Rule of Law',
    'log_Production_Value':                         'NR Production (log, pc)',
    'Trade (% of GDP)':                             'Trade (% GDP)',
    'log_HCI_x_log_Production':                     'HCI × Production',
    'log_GFCF_x_log_Production':                    'GFCF × Production',
    'ECI_lag1':                                     'ECI (t−1)',
    'log_HCI_lag1':                                 'Human Capital (t−1)',
    'log_GFCF_lag1':                                'GFCF (t−1)',
    'Political stability — estimate_lag1':          'Political Stability (t−1)',
    'Rule of law index_lag1':                       'Rule of Law (t−1)',
    'log_Production_Value_lag1':                    'NR Production (t−1)',
    'Trade (% of GDP)_lag1':                        'Trade (t−1)',
    'log_HCI_x_log_Production_lag1':                'HCI × Production (t−1)',
    'log_GFCF_x_log_Production_lag1':               'GFCF × Production (t−1)',
    'Hydrocarbons_Dominant':                        'Hydrocarbons dominant',
    'Subsoil_Metals_Dominant':                      'Subsoil metals dominant',
    'Precious_Metals_Dominant':                     'Precious metals dominant',
    'Access to electricity (% of population)':      'Electricity access',
    'd_log_HCI':                                    'Δ Human Capital (log)',
    'd_log_GFCF':                                   'Δ GFCF (log)',
    'd_Political stability — estimate':             'Δ Political Stability',
    'd_Rule of law index':                          'Δ Rule of Law',
    'd_log_Production_Value':                       'Δ NR Production (log, pc)',
    'd_Trade (% of GDP)':                           'Δ Trade (% GDP)',
    'd_Electricity':                                'Δ Electricity access',
    'd_log_HCI_x_log_Production':                   'Δ HCI × Production',
    'd_log_GFCF_x_log_Production':                  'Δ GFCF × Production',
}

# ── Variable row order in table ───────────────────────────────────────────────
ROW_ORDER = [
    # Levels
    'log_HCI', 'log_GFCF',
    'Political stability — estimate', 'Rule of law index',
    'log_Production_Value', 'Trade (% of GDP)',
    'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    # Lagged levels
    'log_HCI_lag1', 'log_GFCF_lag1',
    'Political stability — estimate_lag1', 'Rule of law index_lag1',
    'log_Production_Value_lag1', 'Trade (% of GDP)_lag1',
    'log_HCI_x_log_Production_lag1', 'log_GFCF_x_log_Production_lag1',
    # First differences
    'd_log_HCI', 'd_log_GFCF',
    'd_Political stability — estimate', 'd_Rule of law index',
    'd_log_Production_Value', 'd_Trade (% of GDP)',
    'd_log_HCI_x_log_Production', 'd_log_GFCF_x_log_Production',
    'd_Electricity',
    # Shared
    'ECI_lag1',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant', 'Precious_Metals_Dominant',
    'Access to electricity (% of population)',
    'const',
]

MODEL_KEYS   = ['M1', 'M2', '3a', '3b', '3c', '3d', '3e']
MODEL_LABELS = {
    'M1': 'M1 Kitchen-sink',
    'M2': 'M2 AR baseline',
    '3a': '3a Base',
    '3b': '3b + ECI(t−1)',
    '3c': '3c All lagged',
    '3d': '3d Extended',
    '3e': '3e First diff.',
}

print('Label maps defined')

Label maps defined


In [9]:
# ── Build wide table: coef(SE)*** per model ───────────────────────────────────
def fmt_cell(coef, se, stars_str):
    if pd.isna(coef):
        return ''
    return f'{coef:.3f}{stars_str}<br><span style="color:#555;font-size:0.85em">({se:.3f})</span>'


# Collect all variables across models
all_vars_present = []
for v in ROW_ORDER:
    for mk in MODEL_KEYS:
        if mk not in ALL_RESULTS:
            continue
        _, tbl, _, _ = ALL_RESULTS[mk]
        if v in tbl['Variable'].values and v not in all_vars_present:
            all_vars_present.append(v)
            break

rows = []
for v in all_vars_present:
    label = DISPLAY_LABELS.get(v, v)
    row = {'Variable': label}
    for mk in MODEL_KEYS:
        if mk not in ALL_RESULTS:
            row[mk] = ''
            continue
        _, tbl, _, _ = ALL_RESULTS[mk]
        match = tbl[tbl['Variable'] == v]
        if len(match) == 0:
            row[mk] = ''
        else:
            r = match.iloc[0]
            row[mk] = fmt_cell(r['Coef'], r['SE'], r['Stars'])
    rows.append(row)

wide = pd.DataFrame(rows)
print(f'Table: {len(wide)} variables × {len(MODEL_KEYS)} models')

Table: 31 variables × 7 models

In [10]:
# ── Build and save styled HTML table ─────────────────────────────────────────
col_headers = ['Variable'] + [MODEL_LABELS[mk] for mk in MODEL_KEYS]
dv_headers  = [''] + [
    ALL_RESULTS[mk][2] if mk in ALL_RESULTS else '' for mk in MODEL_KEYS
]

HTML_CSS = """
<style>
  @import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@300;400;500;600&display=swap');
  body { font-family: 'IBM Plex Sans', Arial, sans-serif; background: #fafafa; }
  .reg-table {
    border-collapse: collapse;
    width: 100%;
    font-size: 13px;
    color: #1a1a2e;
    margin: 24px auto;
    max-width: 1100px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.08);
    background: #fff;
  }
  .reg-table thead tr:first-child th {
    background: #1a1a2e;
    color: #fff;
    font-weight: 600;
    padding: 10px 12px;
    text-align: center;
    border-bottom: 2px solid #1a1a2e;
    white-space: nowrap;
  }
  .reg-table thead tr:first-child th:first-child {
    text-align: left;
  }
  .reg-table thead tr.dv-row th {
    background: #2e2e4e;
    color: #c8c8e0;
    font-weight: 400;
    font-style: italic;
    font-size: 11px;
    padding: 4px 12px;
    text-align: center;
    border-bottom: 1px solid #555;
  }
  .reg-table tbody tr td {
    padding: 7px 12px;
    text-align: center;
    border-bottom: 1px solid #ebebeb;
    vertical-align: top;
    line-height: 1.4;
  }
  .reg-table tbody tr td:first-child {
    text-align: left;
    font-weight: 500;
    white-space: nowrap;
    padding-left: 14px;
  }
  .reg-table tbody tr:nth-child(even) { background: #f5f5fa; }
  .reg-table tbody tr:hover { background: #eef0fc; }
  .reg-table tfoot td {
    padding: 8px 12px;
    font-size: 12px;
    color: #555;
    border-top: 2px solid #1a1a2e;
    text-align: center;
    background: #f0f0f8;
  }
  .reg-table tfoot td:first-child { text-align: left; font-weight: 500; color: #1a1a2e; }
  .note { max-width: 1100px; margin: 0 auto 16px; font-size: 11.5px;
          color: #666; font-style: italic; }
  sup { font-size: 0.75em; color: #c0392b; font-weight: 600; }
</style>
"""

def build_html_table(wide_df, model_keys, all_results, col_headers, dv_headers):
    html = HTML_CSS
    html += '<table class="reg-table">\n'
    
    # Header row
    html += '<thead>\n<tr>\n'
    for h in col_headers:
        html += f'  <th>{h}</th>\n'
    html += '</tr>\n'
    
    # DV sub-header row
    html += '<tr class="dv-row">\n'
    for dv in dv_headers:
        html += f'  <th>Dep. var: {dv}</th>\n' if dv else '<th></th>\n'
    html += '</tr>\n</thead>\n'
    
    # Body
    html += '<tbody>\n'
    for _, row in wide_df.iterrows():
        html += '<tr>\n'
        html += f'  <td>{row["Variable"]}</td>\n'
        for mk in model_keys:
            cell = row.get(mk, '')
            html += f'  <td>{cell}</td>\n'
        html += '</tr>\n'
    html += '</tbody>\n'
    
    # Footer: N and R²
    html += '<tfoot>\n<tr>\n  <td>Observations</td>\n'
    for mk in model_keys:
        if mk in all_results:
            res, *_ = all_results[mk]
            html += f'  <td>{int(res.nobs):,}</td>\n'
        else:
            html += '  <td></td>\n'
    html += '</tr>\n<tr>\n  <td>R²</td>\n'
    for mk in model_keys:
        if mk in all_results:
            res, *_ = all_results[mk]
            html += f'  <td>{res.rsquared:.3f}</td>\n'
        else:
            html += '  <td></td>\n'
    html += '</tr>\n</tfoot>\n</table>\n'
    
    html += ('<p class="note">'
             'Clustered standard errors by country in parentheses (Model 1: by cluster group). '
             '*** p&lt;0.01, ** p&lt;0.05, * p&lt;0.10. '
             'Interaction terms are mean-centred before multiplication. '
             'Model 3e differences all time-varying regressors; ECI(t−1) is an error-correction term.'
             '</p>')
    return html


table_html = build_html_table(wide, MODEL_KEYS, ALL_RESULTS, col_headers, dv_headers)

out_path = os.path.join(OUT, 'regression_table_main.html')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write('<html><head><meta charset="utf-8"></head><body>\n')
    f.write(table_html)
    f.write('</body></html>')
print(f'Saved: {out_path}')

Saved: Graphics/NB6b/regression_table_main.html


## 6  Coefficient forest plot (Models 3a–3d)

In [11]:
FOREST_VARS = [
    'log_HCI', 'log_GFCF',
    'Political stability — estimate', 'Rule of law index',
    'log_Production_Value', 'Trade (% of GDP)',
    'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    'ECI_lag1',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant',
    'Access to electricity (% of population)',
]

# Model 3e uses first-differenced regressors — exclude from forest
FOREST_ORDER = ['3a', '3b', '3c', '3d']
FOREST_LABELS = {
    '3a': '3a Base', '3b': '3b + ECI(t−1)',
    '3c': '3c All lagged', '3d': '3d Extended',
}
SPEC_COLORS = ['#4a6fa5', '#c23a3a', '#2e7d4a', '#d4853b']

y_labels = [DISPLAY_LABELS.get(v, v) for v in FOREST_VARS]
y_pos    = {v: i for i, v in enumerate(y_labels)}
n_specs  = len(FOREST_ORDER)
offsets  = np.linspace(-0.25, 0.25, n_specs)

fig = go.Figure()
fig.add_vline(x=0, line=dict(color='#888', width=1.2, dash='dash'))

for i, mk in enumerate(FOREST_ORDER):
    if mk not in ALL_RESULTS:
        continue
    _, tbl, _, _ = ALL_RESULTS[mk]
    col = SPEC_COLORS[i]
    lbl = FOREST_LABELS[mk]

    matched = tbl[tbl['Variable'].isin(FOREST_VARS)].copy()
    matched = matched.assign(
        Label=matched['Variable'].map(lambda v: DISPLAY_LABELS.get(v, v))
    )
    matched = matched.assign(y=matched['Label'].map(y_pos) + offsets[i])

    fig.add_trace(go.Scatter(
        x=matched['Coef'], y=matched['y'],
        mode='markers',
        marker=dict(size=8, color=col, line=dict(color='white', width=1.2)),
        error_x=dict(
            type='data', symmetric=False,
            array=(matched['CI_hi'] - matched['Coef']).values,
            arrayminus=(matched['Coef'] - matched['CI_lo']).values,
            color=col, thickness=1.8, width=4,
        ),
        name=lbl,
        hovertemplate='%{text}<br>b=%{x:.4f}<extra>' + lbl + '</extra>',
        text=matched['Label'],
    ))

n_vars = len(y_labels)
fig.update_layout(
    font=dict(family=FONT, color=NAVY),
    paper_bgcolor=BG, plot_bgcolor=BG,
    width=1100, height=max(500, n_vars * 55 + 200),
    margin=dict(l=60, r=60, t=60, b=60),
    xaxis=dict(
        title='Coefficient (95% CI, clustered SE by country)',
        gridcolor=GRID, gridwidth=0.5, zeroline=False,
    ),
    yaxis=dict(
        tickvals=list(y_pos.values()),
        ticktext=list(y_pos.keys()),
        tickfont=dict(size=11),
        showgrid=True, gridcolor=GRID, gridwidth=0.5,
        range=[-0.6, n_vars - 0.4],
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.02,
        xanchor='center', x=0.5, font=dict(size=10),
        bgcolor='rgba(255,255,255,0.0)',
    ),
)
save_html(fig, 'coef_forest_3a_3d', cdn=True)
fig.show()

Saved: Graphics/NB6b/coef_forest_3a_3d.html


## 7  VIF diagnostics (Models 3a–3d)

In [12]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def vif_table(df_in, var_list):
    """Return VIF for each variable in var_list, computed on non-missing rows."""
    sub = df_in[var_list].dropna()
    mat = sub.values.astype(float)
    vifs = [variance_inflation_factor(mat, i) for i in range(mat.shape[1])]
    return (
        pd.DataFrame({'Feature': var_list, 'VIF': vifs})
          .sort_values('VIF', ascending=False)
          .reset_index(drop=True)
    ), len(sub)


def _bar_color(vif):
    if vif > 10: return '#c23a3a'
    if vif > 5:  return '#d4853b'
    return '#4a6fa5'


def plot_vif(title, vif_df, n_obs, filename):
    vif_sorted = vif_df.sort_values('VIF')
    colors = [_bar_color(v) for v in vif_sorted['VIF']]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=vif_sorted['VIF'],
        y=vif_sorted['Feature'].map(lambda v: DISPLAY_LABELS.get(v, v)),
        orientation='h',
        marker_color=colors,
        text=[f'{v:.2f}' for v in vif_sorted['VIF']],
        textposition='outside',
        textfont=dict(size=11, family=FONT, color=NAVY),
        cliponaxis=False,
    ))

    x_max = max(vif_sorted['VIF'].max() * 1.25, 12)
    for thresh, label, col in [(5, 'VIF = 5', '#d4853b'), (10, 'VIF = 10', '#c23a3a')]:
        fig.add_vline(x=thresh, line_dash='dash', line_color=col, line_width=1.5,
                      annotation_text=label, annotation_font_size=10,
                      annotation_font_color=col, annotation_position='top right')

    fig.update_layout(
        title=dict(text=f'{title}  (N={n_obs:,})',
                   font=dict(size=13, family=FONT, color=NAVY), x=0.01, xanchor='left'),
        xaxis=dict(title='VIF', range=[0, x_max],
                   gridcolor=GRID, tickfont=dict(family=FONT, size=11)),
        yaxis=dict(tickfont=dict(family=FONT, size=11), automargin=True),
        font=dict(family=FONT, color=NAVY),
        paper_bgcolor=BG, plot_bgcolor=BG,
        margin=dict(l=10, r=90, t=50, b=40),
        height=max(300, 30 * len(vif_df) + 90),
        width=760,
        showlegend=False,
    )
    save_html(fig, filename, cdn=True)   # CDN — ~9 KB instead of 4.6 MB
    fig.show()
    return vif_df


# ── VIF specs ────────────────────────────────────────────────────────────────
VIF_SPECS = {
    '3a': ('3a — Parsimonious (no lag)',
           BASE_INDEP + ['log_HCI_x_log_Production', 'log_GFCF_x_log_Production'],
           'vif_3a'),
    '3b': ('3b — Parsimonious + ECI(t−1)',
           BASE_INDEP + ['ECI_lag1', 'log_HCI_x_log_Production', 'log_GFCF_x_log_Production'],
           'vif_3b'),
    '3c': ('3c — All vars lagged',
           LAGGED_BASE + ['ECI_lag1',
                          'log_HCI_x_log_Production_lag1',
                          'log_GFCF_x_log_Production_lag1'],
           'vif_3c'),
    '3d': ('3d — Extended controls + ECI(t−1)',
           BASE_INDEP + EXTRA_CONTROLS + ['ECI_lag1',
                                           'log_HCI_x_log_Production',
                                           'log_GFCF_x_log_Production'],
           'vif_3d'),
}

print('VIF helpers defined')
print('Note: Model 3e excluded — first-differenced regressors are orthogonal to levels by construction')

VIF helpers defined
Note: Model 3e excluded — first-differenced regressors are orthogonal to levels by construction


In [13]:
ALL_VIF = {}
for mk, (title, var_list, fname) in VIF_SPECS.items():
    vdf, n = vif_table(df, var_list)
    ALL_VIF[mk] = vdf
    print(f'\n── {title}  (N={n:,}) ──')
    print(vdf.to_string(index=False, float_format='{:.2f}'.format))
    plot_vif(title, vdf, n, fname)


── 3a — Parsimonious (no lag)  (N=1,324) ──
                       Feature   VIF
                       log_HCI 27.89
                      log_GFCF 14.83
          log_Production_Value  9.58
              Trade (% of GDP)  7.73
             Rule of law index  4.86
Political stability — estimate  2.04
      log_HCI_x_log_Production  1.86
     log_GFCF_x_log_Production  1.37
Saved: Graphics/NB6b/vif_3a.html



── 3b — Parsimonious + ECI(t−1)  (N=1,273) ──
                       Feature   VIF
                       log_HCI 28.13
                      log_GFCF 15.10
          log_Production_Value  9.74
              Trade (% of GDP)  7.87
             Rule of law index  4.86
                      ECI_lag1  2.40
Political stability — estimate  2.36
      log_HCI_x_log_Production  1.85
     log_GFCF_x_log_Production  1.36
Saved: Graphics/NB6b/vif_3b.html



── 3c — All vars lagged  (N=1,270) ──
                            Feature   VIF
                       log_HCI_lag1 28.37
                      log_GFCF_lag1 15.21
          log_Production_Value_lag1  9.57
              Trade (% of GDP)_lag1  7.83
             Rule of law index_lag1  4.90
                           ECI_lag1  2.40
Political stability — estimate_lag1  2.36
      log_HCI_x_log_Production_lag1  1.90
     log_GFCF_x_log_Production_lag1  1.38


Saved: Graphics/NB6b/vif_3c.html



── 3d — Extended controls + ECI(t−1)  (N=1,264) ──
                                Feature   VIF
                                log_HCI 49.19
                  Hydrocarbons_Dominant 21.00
Access to electricity (% of population) 19.75
                               log_GFCF 18.42
                   log_Production_Value 16.63
                       Trade (% of GDP)  8.22
               Precious_Metals_Dominant  6.85
                      Rule of law index  5.46
                Subsoil_Metals_Dominant  4.30
                               ECI_lag1  3.40
         Political stability — estimate  2.58
               log_HCI_x_log_Production  1.92
              log_GFCF_x_log_Production  1.39
Saved: Graphics/NB6b/vif_3d.html


## 8  Country FE specification (within-estimator)

Within-group demeaning: subtract each country's time-mean from every variable.  
This removes all between-country variation — the source of HCI/GFCF/Trade collinearity.  
`Landlocked` is the only truly time-invariant variable; dominance dummies vary across years and are retained.

In [14]:
def run_fe(df_in, dv, regressors, group='Country Code'):
    """
    OLS with country fixed effects via within-group demeaning.
    Adds back grand mean so the constant is meaningful and R² is within-R².
    Clustered SE by country.
    """
    all_vars = [dv] + regressors
    sub = df_in[all_vars + [group]].dropna().copy()

    # Demean within country, restore grand mean (Frisch-Waugh within estimator)
    for v in all_vars:
        country_mean = sub.groupby(group)[v].transform('mean')
        grand_mean   = sub[v].mean()
        sub[v] = sub[v] - country_mean + grand_mean

    X = sm.add_constant(sub[regressors])
    y = sub[dv]

    # Check for near-zero-variance columns (fully time-invariant — absorbed by FE)
    absorbed = [c for c in regressors if sub[c].std() < 1e-8]
    if absorbed:
        print(f'  ⚠ Absorbed by FE (zero within-variance): {absorbed}')
        regressors = [v for v in regressors if v not in absorbed]
        X = sm.add_constant(sub[regressors])

    res = sm.OLS(y, X).fit(
        cov_type='cluster',
        cov_kwds={'groups': sub[group]},
    )
    n_countries = sub[group].nunique()
    return res, sub, n_countries


FE_RESULTS = {}
print('FE helper defined')

FE helper defined


In [15]:
# ── Run FE versions of 3a–3d ──────────────────────────────────────────────────
FE_SPECS = {
    'FE-3a': (ECI_COL,
              BASE_INDEP + ['log_HCI_x_log_Production', 'log_GFCF_x_log_Production'],
              'Parsimonious, no lag'),
    'FE-3b': (ECI_COL,
              BASE_INDEP + ['ECI_lag1',
                            'log_HCI_x_log_Production', 'log_GFCF_x_log_Production'],
              'Parsimonious + ECI(t−1)'),
    'FE-3c': (ECI_COL,
              LAGGED_BASE + ['ECI_lag1',
                             'log_HCI_x_log_Production_lag1',
                             'log_GFCF_x_log_Production_lag1'],
              'All vars lagged'),
    'FE-3d': (ECI_COL,
              BASE_INDEP + EXTRA_CONTROLS + ['ECI_lag1',
                                              'log_HCI_x_log_Production',
                                              'log_GFCF_x_log_Production'],
              'Extended controls'),
}

for mk, (dv, regs, label) in FE_SPECS.items():
    res, sub, n_c = run_fe(df, dv, regs)
    FE_RESULTS[mk] = (res, to_tbl(res), dv, label)
    print(f'{mk} ({label})  N={int(res.nobs):,}  countries={n_c}  Within-R²={res.rsquared:.4f}')

FE-3a (Parsimonious, no lag)  N=1,324  countries=54  Within-R²=0.0901
FE-3b (Parsimonious + ECI(t−1))  N=1,273  countries=54  Within-R²=0.2674
FE-3c (All vars lagged)  N=1,270  countries=54  Within-R²=0.2663
FE-3d (Extended controls)  N=1,264  countries=54  Within-R²=0.2853


In [16]:
# ── VIF on within-demeaned regressors ─────────────────────────────────────────
print('VIF on within-group demeaned data (country FE)\n')

FE_VIF = {}
for mk, (dv, regs, label) in FE_SPECS.items():
    all_vars = [dv] + regs
    sub = df[all_vars + ['Country Code']].dropna().copy()
    for v in all_vars:
        cm = sub.groupby('Country Code')[v].transform('mean')
        sub[v] = sub[v] - cm

    # Drop absorbed (zero variance) regressors
    active_regs = [v for v in regs if sub[v].std() > 1e-8]
    vdf, n = vif_table(sub, active_regs)
    FE_VIF[mk] = vdf
    print(f'── {mk}: {label}  (N={n:,}) ──')
    print(vdf.to_string(index=False, float_format='{:.2f}'.format))
    print()
    plot_vif(f'{mk} — {label} (country FE)', vdf, n, f'vif_{mk.lower().replace("-","_")}')

VIF on within-group demeaned data (country FE)



── FE-3a: Parsimonious, no lag  (N=1,324) ──
                       Feature  VIF
          log_Production_Value 1.68
      log_HCI_x_log_Production 1.51
                       log_HCI 1.39
     log_GFCF_x_log_Production 1.35
              Trade (% of GDP) 1.18
                      log_GFCF 1.18
Political stability — estimate 1.08
             Rule of law index 1.05

Saved: Graphics/NB6b/vif_fe_3a.html


── FE-3b: Parsimonious + ECI(t−1)  (N=1,273) ──
                       Feature  VIF
          log_Production_Value 1.71
      log_HCI_x_log_Production 1.53
     log_GFCF_x_log_Production 1.38
                       log_HCI 1.38
              Trade (% of GDP) 1.19
                      log_GFCF 1.18
Political stability — estimate 1.08
                      ECI_lag1 1.08
             Rule of law index 1.06



Saved: Graphics/NB6b/vif_fe_3b.html


── FE-3c: All vars lagged  (N=1,270) ──
                            Feature  VIF
          log_Production_Value_lag1 1.80
      log_HCI_x_log_Production_lag1 1.52
                       log_HCI_lag1 1.42
     log_GFCF_x_log_Production_lag1 1.38
              Trade (% of GDP)_lag1 1.19
                      log_GFCF_lag1 1.19
                           ECI_lag1 1.10
Political stability — estimate_lag1 1.08
             Rule of law index_lag1 1.06



Saved: Graphics/NB6b/vif_fe_3c.html


── FE-3d: Extended controls  (N=1,264) ──
                                Feature  VIF
                  Hydrocarbons_Dominant 1.89
                   log_Production_Value 1.88
                Subsoil_Metals_Dominant 1.82
               Precious_Metals_Dominant 1.71
               log_HCI_x_log_Production 1.56
Access to electricity (% of population) 1.56
                                log_HCI 1.55
              log_GFCF_x_log_Production 1.39
                               log_GFCF 1.22
                       Trade (% of GDP) 1.20
         Political stability — estimate 1.09
                               ECI_lag1 1.09
                      Rule of law index 1.07



Saved: Graphics/NB6b/vif_fe_3d.html


In [17]:
# ── Pooled vs FE comparison table ─────────────────────────────────────────────
POOLED_MAP = {'FE-3a': '3a', 'FE-3b': '3b', 'FE-3c': '3c', 'FE-3d': '3d'}

comp_rows = []
for fe_key, pool_key in POOLED_MAP.items():
    fe_res   = FE_RESULTS[fe_key][0]
    pool_res = ALL_RESULTS[pool_key][0]

    fe_vdf   = FE_VIF.get(fe_key, pd.DataFrame())
    pool_vdf = ALL_VIF.get(pool_key, pd.DataFrame())

    comp_rows.append({
        'Spec':            fe_key.replace('FE-', ''),
        'Pooled N':        int(pool_res.nobs),
        'Pooled R²':       round(pool_res.rsquared, 3),
        'Pooled max VIF':  round(pool_vdf['VIF'].max(), 1) if len(pool_vdf) else np.nan,
        'FE N':            int(fe_res.nobs),
        'Within R²':       round(fe_res.rsquared, 3),
        'FE max VIF':      round(fe_vdf['VIF'].max(), 1) if len(fe_vdf) else np.nan,
    })

comp = pd.DataFrame(comp_rows)
print(comp.to_string(index=False))
comp.to_csv(os.path.join(OUT, 'pooled_vs_fe.csv'), index=False)
print(f'\nSaved: {os.path.join(OUT, "pooled_vs_fe.csv")}')

Spec  Pooled N  Pooled R²  Pooled max VIF  FE N  Within R²  FE max VIF
  3a      1324      0.359            27.9  1324      0.090         1.7
  3b      1273      0.785            28.1  1273      0.267         1.7
  3c      1270      0.788            28.4  1270      0.266         1.8
  3d      1264      0.793            49.2  1264      0.285         1.9

Saved: Graphics/NB6b/pooled_vs_fe.csv


In [18]:

# ── HTML table builder (extended, handles both pooled and FE) ──────────────────

HTML_STYLE = """
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:ital,wght@0,300;0,400;0,500;0,600;1,400&display=swap');

* { box-sizing: border-box; }
body { font-family: 'IBM Plex Sans', Arial, sans-serif; background: #f4f4f8; margin: 0; padding: 24px; }

.reg-wrap {
  max-width: 1200px;
  margin: 0 auto 40px;
}
h2.tbl-title {
  font-size: 15px;
  font-weight: 600;
  color: #1a1a2e;
  margin: 0 0 6px;
  letter-spacing: 0.01em;
}
p.tbl-subtitle {
  font-size: 12px;
  color: #666;
  margin: 0 0 12px;
  font-style: italic;
}

.reg-table {
  border-collapse: collapse;
  width: 100%;
  font-size: 12.5px;
  color: #1a1a2e;
  background: #fff;
  box-shadow: 0 1px 6px rgba(0,0,0,0.09);
  border-radius: 4px;
  overflow: hidden;
}

/* ── Header ── */
.reg-table thead tr.hdr-main th {
  background: #1a1a2e;
  color: #fff;
  font-weight: 600;
  padding: 10px 14px;
  text-align: center;
  white-space: nowrap;
  border-right: 1px solid rgba(255,255,255,0.08);
}
.reg-table thead tr.hdr-main th:first-child {
  text-align: left;
  min-width: 200px;
  border-right: 2px solid rgba(255,255,255,0.15);
}

.reg-table thead tr.hdr-dv th {
  background: #2b2b4a;
  color: #a8aad0;
  font-weight: 400;
  font-style: italic;
  font-size: 11px;
  padding: 4px 14px 5px;
  text-align: center;
  border-right: 1px solid rgba(255,255,255,0.06);
  border-bottom: 2px solid #444470;
}
.reg-table thead tr.hdr-dv th:first-child {
  text-align: left;
  border-right: 2px solid rgba(255,255,255,0.12);
}

/* ── Section dividers ── */
.reg-table tbody tr.section-hdr td {
  background: #f0f0f8;
  color: #444466;
  font-weight: 600;
  font-size: 11px;
  text-transform: uppercase;
  letter-spacing: 0.07em;
  padding: 6px 14px 4px;
  border-top: 1px solid #d8d8e8;
}

/* ── Body rows ── */
.reg-table tbody tr.data-row td {
  padding: 6px 14px;
  text-align: center;
  border-bottom: 1px solid #ebebf2;
  border-right: 1px solid #f0f0f5;
  vertical-align: top;
  line-height: 1.45;
}
.reg-table tbody tr.data-row td:first-child {
  text-align: left;
  padding-left: 20px;
  font-weight: 400;
  border-right: 2px solid #e0e0ee;
  white-space: nowrap;
  color: #2a2a40;
}
.reg-table tbody tr.data-row:nth-child(even) { background: #fafafd; }
.reg-table tbody tr.data-row:hover { background: #eef0fc; }

.coef  { font-weight: 500; }
.se    { color: #666; font-size: 0.84em; display: block; margin-top: 1px; }
.stars { color: #c0392b; font-weight: 700; font-size: 0.8em; vertical-align: super; }
.empty { color: #ccc; }

/* ── Footer ── */
.reg-table tfoot tr td {
  padding: 7px 14px;
  font-size: 12px;
  text-align: center;
  border-right: 1px solid #e8e8f0;
  background: #f0f0f8;
  color: #333;
}
.reg-table tfoot tr:first-child td { border-top: 2px solid #1a1a2e; }
.reg-table tfoot tr td:first-child {
  text-align: left;
  font-weight: 600;
  color: #1a1a2e;
  border-right: 2px solid #d0d0e0;
}

/* ── Note ── */
.tbl-note {
  font-size: 11.5px;
  color: #777;
  font-style: italic;
  margin: 8px 0 0;
  line-height: 1.5;
}
</style>
"""

# ── Variable sections for the table ──────────────────────────────────────────
SECTIONS = [
    ('Levels', [
        'log_HCI', 'log_GFCF',
        'Political stability — estimate', 'Rule of law index',
        'log_Production_Value', 'Trade (% of GDP)',
        'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    ]),
    ('Lagged levels', [
        'log_HCI_lag1', 'log_GFCF_lag1',
        'Political stability — estimate_lag1', 'Rule of law index_lag1',
        'log_Production_Value_lag1', 'Trade (% of GDP)_lag1',
        'log_HCI_x_log_Production_lag1', 'log_GFCF_x_log_Production_lag1',
    ]),
    ('First differences', [
        'd_log_HCI', 'd_log_GFCF',
        'd_Political stability — estimate', 'd_Rule of law index',
        'd_log_Production_Value', 'd_Trade (% of GDP)',
        'd_log_HCI_x_log_Production', 'd_log_GFCF_x_log_Production',
        'd_Electricity',
    ]),
    ('Shared / controls', [
        'ECI_lag1',
        'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant', 'Precious_Metals_Dominant',
        'Access to electricity (% of population)',
        'const',
    ]),
]


def build_table(results_dict, model_keys, model_labels, title, subtitle, note, filename):
    """
    results_dict: {key: (res, tbl, dv_label, spec_label)}
    """
    html = HTML_STYLE
    html += f'<div class="reg-wrap">\n'
    html += f'<h2 class="tbl-title">{title}</h2>\n'
    if subtitle:
        html += f'<p class="tbl-subtitle">{subtitle}</p>\n'
    html += '<table class="reg-table">\n<thead>\n'

    # Model name row
    html += '<tr class="hdr-main">\n'
    html += '  <th>Variable</th>\n'
    for mk in model_keys:
        html += f'  <th>{model_labels[mk]}</th>\n'
    html += '</tr>\n'

    # DV sub-row
    html += '<tr class="hdr-dv">\n'
    html += '  <th></th>\n'
    for mk in model_keys:
        dv = results_dict[mk][2] if mk in results_dict else ''
        html += f'  <th>Dep. var: {dv}</th>\n'
    html += '</tr>\n</thead>\n<tbody>\n'

    for section_name, var_list in SECTIONS:
        # Check if this section has any cells across any model
        present = [v for v in var_list
                   if any(v in results_dict[mk][1]['Variable'].values
                          for mk in model_keys if mk in results_dict)]
        if not present:
            continue

        html += f'<tr class="section-hdr"><td colspan="{1 + len(model_keys)}">{section_name}</td></tr>\n'

        for v in present:
            label = DISPLAY_LABELS.get(v, v)
            html += '<tr class="data-row">\n'
            html += f'  <td>{label}</td>\n'
            for mk in model_keys:
                if mk not in results_dict:
                    html += '  <td class="empty">—</td>\n'
                    continue
                _, tbl, _, _ = results_dict[mk]
                row = tbl[tbl['Variable'] == v]
                if len(row) == 0:
                    html += '  <td class="empty">—</td>\n'
                else:
                    r = row.iloc[0]
                    s = r['Stars']
                    html += (f'  <td>'
                             f'<span class="coef">{r["Coef"]:.3f}'
                             f'<span class="stars">{s}</span></span>'
                             f'<span class="se">({r["SE"]:.3f})</span>'
                             f'</td>\n')
            html += '</tr>\n'

    html += '</tbody>\n<tfoot>\n'

    # N row
    html += '<tr>\n  <td>Observations</td>\n'
    for mk in model_keys:
        if mk in results_dict:
            n = int(results_dict[mk][0].nobs)
            html += f'  <td>{n:,}</td>\n'
        else:
            html += '  <td></td>\n'
    html += '</tr>\n'

    # R² row
    r2_label = 'Within R²' if any('FE' in mk for mk in model_keys) else 'R²'
    html += f'<tr>\n  <td>{r2_label}</td>\n'
    for mk in model_keys:
        if mk in results_dict:
            r2 = results_dict[mk][0].rsquared
            html += f'  <td>{r2:.3f}</td>\n'
        else:
            html += '  <td></td>\n'
    html += '</tr>\n</tfoot>\n</table>\n'

    if note:
        html += f'<p class="tbl-note">{note}</p>\n'
    html += '</div>\n'

    path = os.path.join(OUT, filename)
    with open(path, 'w', encoding='utf-8') as f:
        f.write('<html><head><meta charset="utf-8"></head><body>\n')
        f.write(html)
        f.write('</body></html>')
    print(f'Saved: {path}')


print('Builder defined')


Builder defined


In [19]:
NOTE_POOLED = (
    'Clustered standard errors by country in parentheses (Model M1: by cluster group). '
    '*** p&lt;0.01 &nbsp; ** p&lt;0.05 &nbsp; * p&lt;0.10. '
    'Interaction terms are mean-centred before multiplication. '
    'Model 3e differences all time-varying regressors; ECI(t−1) is an error-correction term.'
)
NOTE_FE = (
    'Country fixed effects via within-group demeaning. '
    'Clustered standard errors by country in parentheses. '
    'R² is the within-country R². '
    '*** p&lt;0.01 &nbsp; ** p&lt;0.05 &nbsp; * p&lt;0.10. '
    'Interaction terms are mean-centred before multiplication.'
)
NOTE_FULL = (
    'Columns 3a–3d: pooled OLS; columns FE-3a–FE-3d: same specification with country fixed effects '
    '(within-group demeaning). Clustered SE by country. '
    'R² for pooled models; within-R² for FE models. '
    '*** p&lt;0.01 &nbsp; ** p&lt;0.05 &nbsp; * p&lt;0.10.'
)

POOLED_LABELS = {
    'M1': 'M1 Kitchen-sink', 'M2': 'M2 AR baseline',
    '3a': '3a Base', '3b': '3b + ECI(t−1)',
    '3c': '3c All lagged', '3d': '3d Extended', '3e': '3e First diff.',
}
FE_LABELS = {
    'FE-3a': 'FE-3a Base', 'FE-3b': 'FE-3b + ECI(t−1)',
    'FE-3c': 'FE-3c All lagged', 'FE-3d': 'FE-3d Extended',
}
FULL_LABELS = {
    '3a': '3a Pooled', '3b': '3b Pooled', '3c': '3c Pooled', '3d': '3d Pooled',
    'FE-3a': 'FE-3a', 'FE-3b': 'FE-3b', 'FE-3c': 'FE-3c', 'FE-3d': 'FE-3d',
}

# ── Table 1: Pooled OLS (all seven models) ───────────────────────────────────
build_table(
    ALL_RESULTS,
    model_keys   = ['M1', 'M2', '3a', '3b', '3c', '3d', '3e'],
    model_labels = POOLED_LABELS,
    title        = 'Pooled OLS — all specifications',
    subtitle     = 'OLS with clustered standard errors by country (M1: by cluster). 54-country sample, 1995–2019.',
    note         = NOTE_POOLED,
    filename     = 'table_pooled.html',
)

# ── Table 2: Country FE ───────────────────────────────────────────────────────
build_table(
    FE_RESULTS,
    model_keys   = ['FE-3a', 'FE-3b', 'FE-3c', 'FE-3d'],
    model_labels = FE_LABELS,
    title        = 'Country FE — within-estimator (Models 3a–3d)',
    subtitle     = 'Country fixed effects via within-group demeaning. 54 countries, 1995–2019.',
    note         = NOTE_FE,
    filename     = 'table_fe.html',
)

# ── Table 3: Side-by-side pooled vs FE (parsimonious specs only) ──────────────
combined = {**ALL_RESULTS, **FE_RESULTS}
build_table(
    combined,
    model_keys   = ['3a', 'FE-3a', '3b', 'FE-3b', '3c', 'FE-3c', '3d', 'FE-3d'],
    model_labels = FULL_LABELS,
    title        = 'Pooled OLS vs Country FE — side by side',
    subtitle     = 'Each pair shows the same specification: pooled OLS (left) and country FE within-estimator (right).',
    note         = NOTE_FULL,
    filename     = 'table_pooled_vs_fe.html',
)

Saved: Graphics/NB6b/table_pooled.html
Saved: Graphics/NB6b/table_fe.html
Saved: Graphics/NB6b/table_pooled_vs_fe.html


In [20]:
SPECS_HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Model Specifications</title>
<script>
  window.MathJax = {
    tex: { inlineMath: [['$','$']], displayMath: [['$$','$$']], tags: 'ams' },
    options: { skipHtmlTags: ['script','noscript','style','textarea'] }
  };
</script>
<script async src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js"></script>
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:ital,wght@0,300;0,400;0,500;0,600;1,400&family=IBM+Plex+Mono:wght@400;500&display=swap');

* { box-sizing: border-box; margin: 0; padding: 0; }
body {
  font-family: 'IBM Plex Sans', Arial, sans-serif;
  background: #f4f4f8;
  color: #1a1a2e;
  line-height: 1.65;
  padding: 40px 24px 60px;
}

.page { max-width: 960px; margin: 0 auto; }

h1 {
  font-size: 20px; font-weight: 600; color: #1a1a2e;
  border-bottom: 3px solid #1a1a2e;
  padding-bottom: 10px; margin-bottom: 6px;
}
.doc-subtitle {
  font-size: 13px; color: #777; font-style: italic; margin-bottom: 36px;
}

/* ── Section headers ── */
h2 {
  font-size: 14px; font-weight: 600; letter-spacing: 0.06em;
  text-transform: uppercase; color: #444466;
  margin: 44px 0 18px;
  padding-bottom: 5px;
  border-bottom: 1px solid #d0d0e0;
}

/* ── Model cards ── */
.model-card {
  background: #fff;
  border-radius: 5px;
  box-shadow: 0 1px 5px rgba(0,0,0,0.08);
  margin-bottom: 20px;
  overflow: hidden;
}
.model-header {
  display: flex;
  align-items: baseline;
  gap: 14px;
  background: #1a1a2e;
  padding: 10px 18px;
}
.model-label {
  font-size: 13px; font-weight: 700; color: #fff;
  white-space: nowrap;
}
.model-name {
  font-size: 12px; color: #a8aad0; font-style: italic;
}
.model-body { padding: 16px 20px 18px; }

.eq-block {
  background: #f7f7fc;
  border-left: 3px solid #1a1a2e;
  border-radius: 0 4px 4px 0;
  padding: 14px 20px;
  margin: 10px 0 14px;
  font-size: 14px;
  overflow-x: auto;
}
.model-note {
  font-size: 12.5px; color: #555; line-height: 1.6;
}
.model-note strong { color: #1a1a2e; font-weight: 600; }
.model-note .tag {
  display: inline-block;
  background: #e8e8f4;
  color: #3a3a60;
  font-size: 11px;
  font-weight: 500;
  border-radius: 3px;
  padding: 1px 6px;
  margin-right: 4px;
}

/* ── Diff table ── */
.diff-table {
  width: 100%; border-collapse: collapse;
  font-size: 12.5px; margin-top: 4px;
  background: #fff;
  box-shadow: 0 1px 5px rgba(0,0,0,0.07);
  border-radius: 5px; overflow: hidden;
}
.diff-table thead th {
  background: #1a1a2e; color: #fff;
  font-weight: 600; padding: 9px 14px;
  text-align: left;
}
.diff-table thead th:first-child { width: 18%; }
.diff-table tbody td {
  padding: 8px 14px;
  border-bottom: 1px solid #ebebf2;
  vertical-align: top; line-height: 1.5;
}
.diff-table tbody td:first-child {
  font-weight: 600; color: #3a3a60;
  white-space: nowrap;
  border-right: 2px solid #e0e0ee;
}
.diff-table tbody tr:nth-child(even) { background: #fafafd; }
.diff-table tbody tr:hover { background: #eef0fc; }

.pro  { color: #2e7d4a; font-weight: 600; }
.con  { color: #c23a3a; font-weight: 600; }

/* ── Variable legend ── */
.legend {
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
  gap: 6px 24px;
  background: #fff;
  border-radius: 5px;
  box-shadow: 0 1px 5px rgba(0,0,0,0.07);
  padding: 18px 20px;
  font-size: 12px;
}
.legend-item { display: flex; gap: 10px; align-items: baseline; }
.legend-sym {
  font-family: 'IBM Plex Mono', monospace;
  font-size: 11.5px; color: #3a3a60; font-weight: 500;
  white-space: nowrap; min-width: 130px;
}
.legend-desc { color: #555; line-height: 1.4; }
</style>
</head>
<body>
<div class="page">

<h1>Regression Specifications</h1>
<p class="doc-subtitle">
  54-country sample &nbsp;·&nbsp; 1995–2019 &nbsp;·&nbsp;
  OLS with clustered standard errors by country &nbsp;·&nbsp;
  All interaction terms are mean-centred before multiplication.
</p>

<!-- ══════════════════════════════════════════════════════════════ -->
<h2>Pooled OLS models</h2>

<!-- M1 -->
<div class="model-card">
  <div class="model-header">
    <span class="model-label">M1</span>
    <span class="model-name">Kitchen-sink OLS &nbsp;·&nbsp; Dep. var: log(ECI)</span>
  </div>
  <div class="model-body">
    <div class="eq-block">
      $$\log(\text{ECI}_{it}) = \alpha + \boldsymbol{\beta}'\mathbf{X}_{it} + \varepsilon_{it}$$
    </div>
    <p class="model-note">
      <strong>Purpose:</strong> unrestricted benchmark with the full variable set (~40 regressors)
      covering resource dependence, macroeconomic conditions, governance, human development, and finance.
      Clustered SE by cluster group.<br>
      <span class="tag">DV: log ECI</span>
      <span class="tag">SE: clustered by cluster group</span>
      <span class="tag">~40 regressors</span>
    </p>
  </div>
</div>

<!-- M2 -->
<div class="model-card">
  <div class="model-header">
    <span class="model-label">M2</span>
    <span class="model-name">AR baseline &nbsp;·&nbsp; Dep. var: ECI</span>
  </div>
  <div class="model-body">
    <div class="eq-block">
      $$\text{ECI}_{it} = \alpha + \rho\,\text{ECI}_{i,t-1} + \varepsilon_{it}$$
    </div>
    <p class="model-note">
      <strong>Purpose:</strong> pure autoregressive benchmark. Establishes how much of ECI variation is
      explained by inertia alone, before adding substantive regressors.
      Any model that fails to substantially exceed this R² offers little explanatory power beyond persistence.<br>
      <span class="tag">DV: ECI</span>
      <span class="tag">1 regressor</span>
    </p>
  </div>
</div>

<!-- 3a -->
<div class="model-card">
  <div class="model-header">
    <span class="model-label">3a</span>
    <span class="model-name">Parsimonious — no lag &nbsp;·&nbsp; Dep. var: ECI</span>
  </div>
  <div class="model-body">
    <div class="eq-block">
      $$\text{ECI}_{it} = \alpha
        + \beta_1 \log(\text{HCI}_{it})
        + \beta_2 \log(\text{GFCF}_{it})
        + \beta_3 \text{PolStab}_{it}
        + \beta_4 \text{RoL}_{it}
        + \beta_5 \log(\text{NRProd}_{it})
        + \beta_6 \text{Trade}_{it}
        + \gamma_1 \widetilde{\text{HCI}}_{it} \times \widetilde{\text{NRProd}}_{it}
        + \gamma_2 \widetilde{\text{GFCF}}_{it} \times \widetilde{\text{NRProd}}_{it}
        + \varepsilon_{it}$$
    </div>
    <p class="model-note">
      <strong>Purpose:</strong> parsimonious specification with the core theoretical variables.
      Interaction terms test whether the effect of human and physical capital on complexity
      is moderated by natural resource dependence (resource-curse heterogeneity).
      No lagged ECI — estimates the <em>level</em> association, not the dynamic adjustment.<br>
      <span class="tag">DV: ECI</span>
      <span class="tag">8 regressors</span>
      <span class="tag">no ECI lag</span>
    </p>
  </div>
</div>

<!-- 3b -->
<div class="model-card">
  <div class="model-header">
    <span class="model-label">3b</span>
    <span class="model-name">Parsimonious + ECI(t−1) &nbsp;·&nbsp; Dep. var: ECI</span>
  </div>
  <div class="model-body">
    <div class="eq-block">
      $$\text{ECI}_{it} = \alpha
        + \rho\,\text{ECI}_{i,t-1}
        + \beta_1 \log(\text{HCI}_{it})
        + \beta_2 \log(\text{GFCF}_{it})
        + \beta_3 \text{PolStab}_{it}
        + \beta_4 \text{RoL}_{it}
        + \beta_5 \log(\text{NRProd}_{it})
        + \beta_6 \text{Trade}_{it}
        + \gamma_1 \widetilde{\text{HCI}}_{it} \times \widetilde{\text{NRProd}}_{it}
        + \gamma_2 \widetilde{\text{GFCF}}_{it} \times \widetilde{\text{NRProd}}_{it}
        + \varepsilon_{it}$$
    </div>
    <p class="model-note">
      <strong>Purpose:</strong> adds ECI(t−1) to 3a, controlling for persistence. Coefficients now
      measure the effect of each regressor <em>conditional on last year's complexity level</em> — i.e.
      what drives improvement or deterioration relative to the country's own trajectory.
      $\hat{\rho}$ close to 1 confirms high ECI persistence.<br>
      <span class="tag">DV: ECI</span>
      <span class="tag">9 regressors</span>
      <span class="tag">main preferred spec</span>
    </p>
  </div>
</div>

<!-- 3c -->
<div class="model-card">
  <div class="model-header">
    <span class="model-label">3c</span>
    <span class="model-name">All vars lagged &nbsp;·&nbsp; Dep. var: ECI</span>
  </div>
  <div class="model-body">
    <div class="eq-block">
      $$\text{ECI}_{it} = \alpha
        + \rho\,\text{ECI}_{i,t-1}
        + \beta_1 \log(\text{HCI}_{i,t-1})
        + \beta_2 \log(\text{GFCF}_{i,t-1})
        + \beta_3 \text{PolStab}_{i,t-1}
        + \beta_4 \text{RoL}_{i,t-1}
        + \beta_5 \log(\text{NRProd}_{i,t-1})
        + \beta_6 \text{Trade}_{i,t-1}
        + \gamma_1 \widetilde{\text{HCI}}_{i,t-1} \times \widetilde{\text{NRProd}}_{i,t-1}
        + \gamma_2 \widetilde{\text{GFCF}}_{i,t-1} \times \widetilde{\text{NRProd}}_{i,t-1}
        + \varepsilon_{it}$$
    </div>
    <p class="model-note">
      <strong>Purpose:</strong> lagging all regressors by one period reduces simultaneity bias — today's ECI
      cannot cause last year's HCI or governance. Also mitigates reverse causality.
      Compares to 3b to check whether contemporaneous vs lagged regressors give materially different estimates.<br>
      <span class="tag">DV: ECI</span>
      <span class="tag">9 regressors, all at t−1</span>
      <span class="tag">simultaneity-robust</span>
    </p>
  </div>
</div>

<!-- 3d -->
<div class="model-card">
  <div class="model-header">
    <span class="model-label">3d</span>
    <span class="model-name">Extended controls &nbsp;·&nbsp; Dep. var: ECI</span>
  </div>
  <div class="model-body">
    <div class="eq-block">
      $$\text{ECI}_{it} = \alpha
        + \rho\,\text{ECI}_{i,t-1}
        + \boldsymbol{\beta}'\mathbf{X}_{it}
        + \delta_1 \mathbf{1}[\text{Hydro}_{it}]
        + \delta_2 \mathbf{1}[\text{SubsoilMet}_{it}]
        + \delta_3 \mathbf{1}[\text{PrecMet}_{it}]
        + \delta_4 \text{Elec}_{it}
        + \varepsilon_{it}$$
    </div>
    <p class="model-note">
      <strong>Purpose:</strong> augments 3b with resource-type dominance dummies and electricity access,
      testing whether the <em>type</em> of resource dependence matters beyond the production level, and
      whether infrastructure access is an independent channel. Note: high multicollinearity in pooled OLS
      (max VIF = 49) — FE-3d is the preferred version of this spec.<br>
      <span class="tag">DV: ECI</span>
      <span class="tag">13 regressors</span>
      <span class="tag">use FE version</span>
    </p>
  </div>
</div>

<!-- 3e -->
<div class="model-card">
  <div class="model-header">
    <span class="model-label">3e</span>
    <span class="model-name">First differences (error-correction) &nbsp;·&nbsp; Dep. var: ΔECI</span>
  </div>
  <div class="model-body">
    <div class="eq-block">
      $$\Delta\text{ECI}_{it} = \alpha
        + \delta\,\text{ECI}_{i,t-1}
        + \beta_1 \Delta\log(\text{HCI}_{it})
        + \beta_2 \Delta\log(\text{GFCF}_{it})
        + \beta_3 \Delta\text{PolStab}_{it}
        + \beta_4 \Delta\text{RoL}_{it}
        + \beta_5 \Delta\log(\text{NRProd}_{it})
        + \beta_6 \Delta\text{Trade}_{it}
        + \beta_7 \Delta\text{Elec}_{it}
        + \gamma_1 \Delta(\widetilde{\text{HCI}} \times \widetilde{\text{NRProd}})_{it}
        + \gamma_2 \Delta(\widetilde{\text{GFCF}} \times \widetilde{\text{NRProd}})_{it}
        + \varepsilon_{it}$$
    </div>
    <p class="model-note">
      <strong>Purpose:</strong> proper first-difference specification — both sides are differenced.
      Time-invariant variables (dominance dummies) drop out by construction.
      ECI(t−1) is retained as an <em>error-correction term</em>: a negative $\hat{\delta}$ implies
      that countries farther above their long-run ECI path grow more slowly, consistent with mean-reversion.
      Coefficients measure how year-on-year <em>changes</em> in regressors predict year-on-year
      <em>changes</em> in complexity — a stricter and more causally interpretable test than levels.<br>
      <span class="tag">DV: ΔECI</span>
      <span class="tag">10 regressors, all first-differenced</span>
      <span class="tag">error-correction term</span>
    </p>
  </div>
</div>

<!-- ══════════════════════════════════════════════════════════════ -->
<h2>Country FE models (FE-3a to FE-3d)</h2>

<div class="model-card">
  <div class="model-header">
    <span class="model-label">FE-3a / FE-3b / FE-3c / FE-3d</span>
    <span class="model-name">Within-group estimator &nbsp;·&nbsp; Dep. var: ECI</span>
  </div>
  <div class="model-body">
    <div class="eq-block">
      $$\text{ECI}_{it} = \mu_i + \boldsymbol{\beta}'\mathbf{X}_{it} + \varepsilon_{it}$$
      <br>
      $$\Longleftrightarrow \quad
        \underbrace{(\text{ECI}_{it} - \overline{\text{ECI}}_i)}_{\text{within-country deviation}}
        = \boldsymbol{\beta}'
        \underbrace{(\mathbf{X}_{it} - \overline{\mathbf{X}}_i)}_{\text{within-country deviation}}
        + (\varepsilon_{it} - \bar{\varepsilon}_i)$$
    </div>
    <p class="model-note">
      Each FE model uses the same regressor set as its pooled counterpart (3a–3d) but absorbs
      country-level intercepts $\mu_i$ via within-group demeaning. The variable set $\mathbf{X}_{it}$
      corresponds to the respective pooled specification.<br><br>
      <strong>Implementation:</strong> for each variable $v_{it}$, subtract the country time-mean
      $\bar{v}_i$ and add back the grand mean, then run OLS with a constant.
      This is the Frisch–Waugh–Lovell within-estimator, numerically identical to including
      $N-1$ country dummies.
    </p>
  </div>
</div>

<!-- ══════════════════════════════════════════════════════════════ -->
<h2>Pooled OLS vs Country FE — key differences</h2>

<table class="diff-table">
  <thead>
    <tr>
      <th>Dimension</th>
      <th>Pooled OLS</th>
      <th>Country FE (within-estimator)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Variation used</td>
      <td>Both <em>between</em>-country (cross-sectional) and <em>within</em>-country (over time) variation.</td>
      <td>Only <em>within</em>-country, over-time variation. Cross-sectional differences across countries are removed.</td>
    </tr>
    <tr>
      <td>Unobserved heterogeneity</td>
      <td>Assumes no country-specific unobservables correlated with regressors (strict exogeneity). Violation → omitted variable bias.</td>
      <td>Absorbs all time-invariant country characteristics (geography, institutions, culture) whether observed or not. Robust to this class of omitted variables.</td>
    </tr>
    <tr>
      <td>Multicollinearity (VIF)</td>
      <td>High: HCI and GFCF are both development-level proxies correlated in the cross-section. Max VIF = 28–49 across specs.</td>
      <td><span class="pro">Resolved:</span> within-country variation in HCI and GFCF is nearly orthogonal. Max VIF = 1.7–1.9 across all specs.</td>
    </tr>
    <tr>
      <td>Time-invariant variables</td>
      <td><span class="pro">Identified:</span> variables like dominance dummies or Landlocked have non-zero coefficients.</td>
      <td><span class="con">Absorbed:</span> variables with zero within-country variance (e.g. Landlocked) are collinear with the fixed effects and drop out. Dominance dummies survive because they do vary over time.</td>
    </tr>
    <tr>
      <td>What the R² measures</td>
      <td>Share of total ECI variation (across countries and years) explained. Inflated by between-country differences in ECI levels.</td>
      <td><em>Within</em>-R²: share of within-country, year-to-year ECI variation explained. More demanding and more meaningful for causal inference.</td>
    </tr>
    <tr>
      <td>R² in practice</td>
      <td>0.36 – 0.79 depending on spec.</td>
      <td>0.09 – 0.29. Lower because it ignores the easy between-country signal — not a sign of worse fit.</td>
    </tr>
    <tr>
      <td>Causal interpretation</td>
      <td>Estimates the association: "countries with higher HCI tend to have higher ECI." Confounded by omitted country-level factors.</td>
      <td>Estimates the within-country effect: "when a country's HCI rises above its own historical average, does ECI follow?" Cleaner causal interpretation.</td>
    </tr>
    <tr>
      <td>Assumption required</td>
      <td>No omitted variable correlated with regressors (strong).</td>
      <td>No <em>time-varying</em> omitted variable correlated with regressors (weaker). Fixed country characteristics are controlled for by construction.</td>
    </tr>
    <tr>
      <td>Preferred for this study</td>
      <td>M1, M2 as benchmarks. 3a–3e as robustness checks.</td>
      <td><span class="pro">FE-3b / FE-3d as main specifications</span> — resolves multicollinearity, controls for unobserved country heterogeneity, and gives within-country causal estimates.</td>
    </tr>
  </tbody>
</table>

<!-- ══════════════════════════════════════════════════════════════ -->
<h2>Variable notation</h2>

<div class="legend">
  <div class="legend-item">
    <span class="legend-sym">$\text{ECI}_{it}$</span>
    <span class="legend-desc">Economic Complexity Index, country $i$, year $t$</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\log(\text{HCI}_{it})$</span>
    <span class="legend-desc">log(1 + Human Capital Index)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\log(\text{GFCF}_{it})$</span>
    <span class="legend-desc">log(1 + Gross Fixed Capital Formation, % GDP)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\text{PolStab}_{it}$</span>
    <span class="legend-desc">Political Stability estimate (WGI)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\text{RoL}_{it}$</span>
    <span class="legend-desc">Rule of Law index (WGI)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\log(\text{NRProd}_{it})$</span>
    <span class="legend-desc">log(1 + NR production value per capita)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\text{Trade}_{it}$</span>
    <span class="legend-desc">Trade openness (% of GDP)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\text{Elec}_{it}$</span>
    <span class="legend-desc">Access to electricity (% of population)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\widetilde{v}_{it}$</span>
    <span class="legend-desc">Mean-centred version of variable $v$: $v_{it} - \bar{v}$</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\mathbf{1}[\cdot]$</span>
    <span class="legend-desc">Indicator dummy (1 if resource type is dominant)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\mu_i$</span>
    <span class="legend-desc">Country fixed effect (absorbed by within-group demeaning)</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\Delta v_{it}$</span>
    <span class="legend-desc">First difference: $v_{it} - v_{i,t-1}$</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\overline{v}_i$</span>
    <span class="legend-desc">Country time-mean: $T^{-1}\sum_t v_{it}$</span>
  </div>
  <div class="legend-item">
    <span class="legend-sym">$\varepsilon_{it}$</span>
    <span class="legend-desc">Error term, clustered by country</span>
  </div>
</div>

</div><!-- .page -->
</body>
</html>
"""

spec_path = os.path.join(OUT, 'model_specs.html')
with open(spec_path, 'w', encoding='utf-8') as f:
    f.write(SPECS_HTML)
print(f'Saved: {spec_path}')


Saved: Graphics/NB6b/model_specs.html
